# Train Symptom Triage Coach LoRA (Colab)

Runtime: **T4 GPU** (Runtime → Change runtime type → T4 GPU).

Before running, in Colab Secrets (key icon), add:

| Name | Value |
|------|-------|
| `HF_TOKEN` | your Hugging Face write token |

**This notebook assumes data/processed/train.jsonl and val.jsonl already exist in the GitHub repo** (generated locally via `src/generate_data.py` and committed).

In [ ]:
!pip install -q torch transformers peft accelerate bitsandbytes datasets trl sentencepiece huggingface-hub jsonschema

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

In [ ]:
!git clone https://github.com/ksolano220/symptom-triage-coach.git
%cd symptom-triage-coach

In [ ]:
import sys
sys.path.insert(0, 'src')
from train import train

trainer = train(
    train_path='data/processed/train.jsonl',
    val_path='data/processed/val.jsonl',
    hub_repo_id='ksolano220/symptom-triage-coach',
)

In [ ]:
import json
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

from schema import SYSTEM_PROMPT

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct', torch_dtype=torch.bfloat16, device_map='auto')
model = PeftModel.from_pretrained(base, 'ksolano220/symptom-triage-coach')
model.eval()

messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': 'I have chest pain when I breathe deeply'},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=500, do_sample=False, pad_token_id=tokenizer.eos_token_id)
gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(json.dumps(json.loads(gen), indent=2))